# Clase 3 — Estar equivocado es un número, y ese número tiene pendiente

**Deep Learning para Cientistas Sociales**

> **La idea de hoy en una línea.** Si podés ponerle un número a *qué tan mal la
> estoy dibujando*, ese número dibuja un paisaje sobre el espacio de las
> perillas. Aprender es soltar una pelota en ese paisaje y dejarla rodar hasta
> el fondo.

---

**Cuando termines este notebook vas a poder:**

1. Escribir en dos líneas de Python las dos funciones de pérdida más comunes
   (**MSE** y **MAE**) y decir en castellano en qué se diferencian.
2. Dibujar el **paisaje de la loss** de tu propio modelo y encontrar el fondo
   del valle a ojo.
3. Calcular una pendiente **con una resta**, sin haber estudiado cálculo, y
   escribir el descenso por gradiente completo en diez líneas.
4. Reconocer, mirando **sólo la curva de loss**, si tu tasa de aprendizaje es
   muy chica, está bien o es demasiado grande.

**Lo que hace falta traer:** las clases 1 y 2. Nada más.

**Cómo se usa este notebook.** Corré las celdas en orden, de arriba hacia abajo
(`Shift + Enter`). Donde diga `# TODO` te toca a vos. Cada ejercicio tiene una
celda de verificación que te dice, en castellano, si te salió y qué mirar si no.
Nada explota: si algo está mal, te lo dice con palabras.

## 0. Preparar el entorno

Esta celda es **idéntica en los diez notebooks del curso**. En Colab clona el
repo y deja todo a mano; en tu máquina no hace prácticamente nada porque ya
tenés los archivos. Corrémela y seguí de largo: no hay nada que entender acá.

In [ ]:
# =============================================================================

# BOOTSTRAP — la misma celda en todos los notebooks del curso. No la edites.

# En Colab: clona el repo del curso e instala lo que falte.

# En tu máquina: encuentra el repo subiendo carpetas y no instala nada.

# =============================================================================

import os

import pathlib

import subprocess

import sys



REPO = "https://github.com/johannperez/dlss"

RAMA = "claude/dl-course-social-scientists-279dd2"

CARPETA = "dlss"





def en_colab():

    """¿Estamos en Google Colab? Se lo preguntamos a Python, no lo adivinamos."""

    try:

        import google.colab  # noqa: F401

        return True

    except ImportError:

        return False





def buscar_raiz(desde=None):

    """Sube carpeta por carpeta hasta encontrar la que tiene `_shared/ssdl`."""

    punto = pathlib.Path(desde or os.getcwd()).resolve()

    for candidata in [punto, *punto.parents]:

        if (candidata / "_shared" / "ssdl").is_dir():

            return candidata

    return None





raiz = buscar_raiz()



if raiz is None and en_colab():

    if not pathlib.Path(CARPETA).exists():

        print("Bajando el repo del curso (una sola vez por sesión)...")

        subprocess.run(["git", "clone", "--depth", "1", "--branch", RAMA,

                        REPO, CARPETA], check=True)

    raiz = buscar_raiz(CARPETA)



if raiz is None:

    raise SystemExit(

        "No encuentro el repo del curso.\n"

        "· En Colab: revisá que la celda haya podido clonar (necesita internet).\n"

        "· En tu máquina: abrí este notebook desde adentro de la carpeta `dlss`."

    )



# Dependencias: en Colab casi todas vienen puestas. Instalamos sólo lo que falte.

for modulo, paquete in [("numpy", "numpy"), ("matplotlib", "matplotlib"),

                        ("torch", "torch")]:

    try:

        __import__(modulo)

    except ImportError:

        print(f"Instalando {paquete}...")

        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete],

                       check=True)



sys.path.insert(0, str(raiz / "_shared"))   # para poder hacer `import ssdl`

os.environ["DLSS_RAIZ"] = str(raiz)         # para que ssdl encuentre los datos



print(f"Listo. Repo del curso en: {raiz}")

### Imports, semilla y una ayudita

`ssdl` es el paquete del curso: la plomería que no queremos repetir en diez
notebooks. La referencia completa está en `_shared/ssdl/README.md`.

Fijamos la semilla en 1983 **siempre**. No hace al modelo mejor: hace a tu
resultado reproducible, que es condición para poder afirmar algo.

La función `revisar()` de abajo es la que usan los ejercicios. **No levanta
excepciones**: te dice con palabras si te salió. Un stack trace rojo no le
enseña nada a nadie.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from ssdl import datos, graficos as g, utils

utils.fijar_semilla(1983)          # la semilla del curso
utils.resumen_del_entorno()


def revisar(condicion, si_va, si_no):
    """Verificación amable: nunca explota, siempre explica.

    >>> revisar(2 + 2 == 4, "sabés sumar", "revisá la suma")
    ✔  sabés sumar
    """
    print(("✔  " if condicion else "✱  ") + (si_va if condicion else si_no))
    return bool(condicion)

---

## 1. La loss: el número gris de la clase 1, ahora con nombre

En la clase 1 movías dos perillas a ojo y mirabas un número que decía «qué tan
mal está esto». Ese número tiene tres nombres, porque este campo no se pone de
acuerdo en nada:

| en inglés | en castellano | qué es |
|---|---|---|
| *loss* | pérdida | un número que mide qué tan mal la estás dibujando |
| *loss function* | función de pérdida | la receta para calcular ese número |
| *cost function* | función de costo | lo mismo, otro nombre |

**Lo importante no es el nombre: es que la receta la elegís vos.** Hay muchas
maneras de resumir «qué tan mal está esto» en un solo número, y cada una le
hace caso a casos distintos. Vamos a ver las dos más comunes.

**La analogía de ciencias sociales.** Elegir la loss es *operacionalizar el
fracaso*. Toda evaluación de política pública tiene que definir qué cuenta como
error: ¿una política de vivienda fracasa por dejar gente afuera, o por darle a
quien no lo necesitaba? Elegir la loss es exactamente esa discusión, escrita en
una línea de código.

**Lo que NO es.** No es una medida objetiva de la verdad. Es una declaración de
qué error te duele más.

In [ ]:
# Los datos de siempre: qué porcentaje de las palabras del discurso presidencial
# de apertura de sesiones se dedicó a "seguridad", año por año. Datos reales.
anios, seguridad = datos.serie_menciones("seguridad", desde=1984, hasta=2019)

# El año, CENTRADO y en décadas. ¿Por qué? Porque si dejamos el año en 1984-2019,
# la perilla de la altura y la de la inclinación quedan pegadas una a la otra y
# el valle de la loss sale como una zanja diagonal. Con el año centrado, sale
# redondo. Volvemos sobre esto al final.
x = (anios - 2001.5) / 10          # -1.75 .. 1.75, en décadas
y = seguridad                      # % de palabras

print(f"{len(anios)} años, de {int(anios[0])} a {int(anios[-1])}")
print(f"x va de {x.min():.2f} a {x.max():.2f} (décadas desde 2001)")
print(f"y va de {y.min():.2f}% a {y.max():.2f}% de las palabras del discurso")

g.plot_serie(anios, seguridad, etiqueta="menciones a seguridad",
             titulo="Lo que vamos a tratar de dibujar con una recta")
plt.show()

### Las dos losses, en dos líneas

Las dos hacen lo mismo: toman los errores de cada caso (la distancia vertical
entre el punto y la recta) y los resumen en un número. La diferencia es **cómo
castigan un error grande**.

- **MSE** (*mean squared error*, error cuadrático medio): eleva cada error al
  cuadrado. Un error de 2 pesa 4; uno de 20 pesa **400**.
- **MAE** (*mean absolute error*, error absoluto medio): toma el error tal
  cual, sin signo. Un error de 20 pesa 20.

Nada más. Todo lo que sigue en esta clase sale de esa diferencia.

In [ ]:
def mse(y_real, y_pred):
    """Error cuadrático medio: castiga muchísimo los errores grandes."""
    return np.mean((y_real - y_pred) ** 2)


def mae(y_real, y_pred):
    """Error absoluto medio: trata a todos los errores igual de mal."""
    return np.mean(np.abs(y_real - y_pred))


# Una recta cualquiera, puesta a ojo (todavía no la buscamos: la ponemos)
a, b = 0.05, 0.15                  # inclinación (por década) y altura
prediccion = a * x + b

print(f"Con la recta y = {a}·x + {b}")
print(f"   MSE = {mse(y, prediccion):.5f}")
print(f"   MAE = {mae(y, prediccion):.5f}")
print()
print("Ojo: NO se comparan entre sí. Están en unidades distintas.")
print("El MSE está en 'puntos al cuadrado' y el MAE en 'puntos'.")
print("Cada uno se compara consigo mismo cuando movés las perillas.")

g.plot_ajuste(anios, seguridad, modelo=lambda t: a * (t - 2001.5) / 10 + b,
              mostrar_errores=True,
              xlabel="año", ylabel="% de palabras",
              etiqueta_modelo=f"mi recta (a={a}, b={b})",
              titulo="Los palitos rojos son los errores. La loss los resume en un número.")
plt.show()

### Ahora metemos un caso extremo

Supongamos que en 2019 el discurso hubiera dedicado a «seguridad» el 1,5 % de
sus palabras: tres veces y media el máximo real de la serie. Puede pasar (un
año excepcional), o puede ser un error de carga. **Todavía no sabemos cuál de
las dos cosas es**, y esa es exactamente la situación en la que vas a estar vos
con tus datos.

Miremos qué le hace ese único caso a cada una de las dos losses.

In [ ]:
# Copiamos la serie y le cambiamos UN valor: el último año
y_con_extremo = y.copy()
y_con_extremo[-1] = 1.50           # el valor real de 2019 era 0.2453

sin_mse, sin_mae = mse(y, prediccion), mae(y, prediccion)
con_mse, con_mae = mse(y_con_extremo, prediccion), mae(y_con_extremo, prediccion)

print(f"{'':12} {'sin el caso':>12} {'con el caso':>12} {'creció':>10}")
print(f"{'MSE':12} {sin_mse:12.5f} {con_mse:12.5f} {con_mse/sin_mse:9.1f}x")
print(f"{'MAE':12} {sin_mae:12.5f} {con_mae:12.5f} {con_mae/sin_mae:9.1f}x")
print()
print("Un solo caso, de 36. Mirá la última columna y volvé a leerla.")

### `TODO 1` — ¿cuál de las dos subió más, y por qué?

**Consigna.** Escribí en la celda de abajo, en un comentario, dos cosas:

1. Cuál de las dos losses creció proporcionalmente más al agregar el caso
   extremo, y **por qué** (una oración, sin fórmulas).
2. Cuál elegirías si supieras que ese caso es **un error de carga**, y cuál si
   supieras que ese caso es **el año que motivó tu investigación**.

**Pista.** Mirá qué le hace el cuadrado a un error de 1,3: `1.3 ** 2 = 1.69`.
Ahora mirá qué le hace a un error de 0,1: `0.1 ** 2 = 0.01`. El cuadrado no
castiga «un poco más» a los errores grandes: los castiga *desproporcionadamente*
más.

**Cómo saber si te salió.** La celda de verificación de abajo compara los dos
crecimientos y te dice el número. Si tu respuesta escrita coincide con lo que
imprime, cerraste.

In [ ]:
# TODO 1: escribí tus dos respuestas acá, en comentarios.
#
# 1. La que creció más fue ......... porque ...........................
#
# 2. Si el caso extremo fuera un error de carga, elegiría ..............
#    Si fuera EL caso que motiva mi investigación, elegiría ............

crecio_mse = con_mse / sin_mse
crecio_mae = con_mae / sin_mae

revisar(crecio_mse > crecio_mae,
        f"El MSE creció {crecio_mse:.1f}x y el MAE sólo {crecio_mae:.1f}x. "
        "El cuadrado es lo que hace la diferencia: un error grande, elevado al "
        "cuadrado, se come la loss entera.",
        "Algo raro pasó con los datos. Volvé a correr la celda del caso extremo.")

# Cuánto del MSE total lo aporta ese único año
errores = y_con_extremo - prediccion
cuota = errores[-1] ** 2 / np.sum(errores ** 2)
print(f"\nEse único año aporta el {cuota:.0%} de TODO el MSE.")
print("Cuando el modelo busque la mejor recta, va a mover el mundo por ese año.")

---

## 2. El paisaje de la loss

Hasta acá teníamos **una** recta y **un** número. Ahora damos el salto
conceptual más importante de la clase, y conviene hacerlo despacio:

> Cada par de valores `(a, b)` **es una recta entera**. Si dibujamos un plano
> donde el eje horizontal es `a` y el vertical es `b`, cada punto de ese plano
> es una recta distinta. Y a cada punto le podemos poner una altura: su loss.

Eso es un **paisaje**. Un mapa topográfico donde lo bajo es «esta recta ajusta
bien» y lo alto es «esta recta es un desastre».

**Ojo con esto.** En ese mapa no hay ningún año y ningún porcentaje. Los ejes
**no** son las variables de tu estudio: son las perillas de tu modelo. Es la
confusión número uno de esta clase y vale la pena decirla en voz alta:
*en el mapa no hay datos, hay rectas.*

Lo construimos con un doble `for`, que es la manera más tonta y más honesta.

In [ ]:
def loss(a, b):
    """El desajuste de la recta y = a·x + b sobre NUESTROS datos."""
    return mse(y, a * x + b)


# La grilla: 120 valores de "a" por 120 valores de "b" = 14.400 rectas
rejilla_a = np.linspace(-0.40, 0.50, 120)
rejilla_b = np.linspace(-0.10, 0.42, 120)

paisaje = np.zeros((len(rejilla_b), len(rejilla_a)))
for i, bb in enumerate(rejilla_b):          # las filas son la altura
    for j, aa in enumerate(rejilla_a):      # las columnas, la inclinación
        paisaje[i, j] = loss(aa, bb)

# ¿Dónde está el punto más bajo de los 14.400 que probamos?
fila, col = np.unravel_index(np.argmin(paisaje), paisaje.shape)
mejor_a, mejor_b = rejilla_a[col], rejilla_b[fila]

print(f"Probamos {paisaje.size:,} rectas.".replace(",", "."))
print(f"La mejor de todas: a = {mejor_a:.4f}, b = {mejor_b:.4f}")
print(f"Su desajuste: {paisaje.min():.6f}")
print()
print("Traducido a castellano sustantivo, esa recta afirma:")
print(f"  «las menciones a seguridad suben {mejor_a:.3f} puntos por década,")
print(f"   y en 2001 valían {mejor_b:.3f} % de las palabras del discurso».")

In [ ]:
# El mapa topográfico, dibujado a mano con matplotlib
fig, ax = g.figura(7.5, 5.5,
                   titulo="El paisaje de la loss: cada punto es una recta entera",
                   xlabel="qué tan empinada (puntos por década)",
                   ylabel="dónde arranca (valor en 2001)")

# contourf = relleno por niveles; los niveles van en escala logarítmica porque
# el paisaje sube muchísimo en los bordes y si no, el valle queda todo de un color
niveles = np.geomspace(paisaje.min(), paisaje.max(), 22)
relleno = ax.contourf(rejilla_a, rejilla_b, paisaje, levels=niveles, cmap="YlOrRd")
ax.contour(rejilla_a, rejilla_b, paisaje, levels=niveles, colors="white",
           linewidths=.5, alpha=.55)
fig.colorbar(relleno, ax=ax, label="desajuste (MSE)")

ax.set_aspect("equal")      # las dos perillas, en la misma escala: si no, un
                            # valle redondo se ve como una elipse y confunde
ax.plot(mejor_a, mejor_b, "o", color=g.OK, markersize=11,
        markeredgecolor="white", markeredgewidth=2, zorder=5)
ax.annotate("el fondo del valle", (mejor_a, mejor_b), xytext=(14, 14),
            textcoords="offset points", color=g.OK, fontweight="bold")
plt.show()

### `TODO 2` — hacé zoom cerca del fondo

**Consigna.** Copiá el bloque de arriba en la celda siguiente y cambiá los
límites de `np.linspace` para mirar de cerca la zona del mínimo. ¿Qué forma
tiene el valle cuando lo mirás de cerca: redondo, alargado, torcido?

**Pista.** Probá con `np.linspace(mejor_a - 0.06, mejor_a + 0.06, 120)` y lo
mismo para `b`. Si querés ver algo divertido, probá después con un rango muy
ancho en `a` y muy angosto en `b`.

**Cómo saber si te salió.** Tenés que ver contornos casi **circulares**.
Guardá esa forma en la memoria: en la última sección vamos a romperla a
propósito y vas a entender por qué el descenso a veces zigzaguea.

In [ ]:
# TODO 2: hacé zoom. Cambiá estos dos rangos y volvé a correr.
zoom_a = np.linspace(mejor_a - 0.06, mejor_a + 0.06, 120)   # TODO: probá otros
zoom_b = np.linspace(mejor_b - 0.06, mejor_b + 0.06, 120)   # TODO: probá otros

zoom = np.array([[loss(aa, bb) for aa in zoom_a] for bb in zoom_b])

fig, ax = g.figura(6.5, 5, titulo="El valle, de cerca",
                   xlabel="qué tan empinada", ylabel="dónde arranca")
ax.contourf(zoom_a, zoom_b, zoom, levels=18, cmap="YlOrRd")
ax.contour(zoom_a, zoom_b, zoom, levels=18, colors="white", linewidths=.5, alpha=.6)
ax.set_aspect("equal")
ax.plot(mejor_a, mejor_b, "o", color=g.OK, markersize=10,
        markeredgecolor="white", markeredgewidth=2)
plt.show()

ancho_a = zoom_a[-1] - zoom_a[0]
revisar(abs(ancho_a) < 0.5,
        "Estás mirando de cerca. Los contornos se ven casi circulares: este "
        "valle es redondo, y eso es una suerte que nos ganamos al centrar el año.",
        "El rango sigue siendo muy ancho: achicá los límites de np.linspace "
        "para acercarte de verdad al fondo.")

---

## 3. La fuerza bruta funciona… y se muere en la línea 3

Acabamos de encontrar la mejor recta probando 14.400. **Funcionó.** Con dos
perillas, probar todo es una estrategia perfectamente razonable.

El problema es que no escala. Y no escala de una manera que vale la pena ver
con números, porque es la razón por la que existe todo lo que viene después.

In [ ]:
# Si probamos 100 valores por perilla, ¿cuántas combinaciones hay que evaluar?
# Los números se van tan rápido de escala que ni siquiera entran en un float:
# hay que trabajar con el EXPONENTE. Eso ya te dice algo.
modelos = [
    ("nuestra recta", 2),
    ("una regresión con 3 variables", 4),   # 3 pendientes + la altura
    ("la red de 8 rampas de la clase 2", 8),
    ("una red chiquita", 100),
    ("el mini-GPT de la clase 8", 1_500_000),
]

# Siendo generosos: mil millones (10^9) de pruebas por segundo
EXP_POR_SEGUNDO = 9
EXP_SEGUNDOS_POR_ANIO = 7.5          # un año tiene ~3,15 x 10^7 segundos
EXP_EDAD_UNIVERSO = 17.64            # ~13.800 millones de años, en segundos


def en_castellano(exp_segundos):
    """Traduce «diez elevado a N segundos» a algo que se pueda decir en voz alta."""
    if exp_segundos < 0.8:
        return "menos de 6 segundos"
    if exp_segundos < 1.8:                       # menos de un minuto
        return f"{10 ** exp_segundos:.0f} segundos"
    if exp_segundos < 4.94:                      # menos de un día
        return f"{10 ** (exp_segundos - 3.56):.0f} horas"
    if exp_segundos < EXP_SEGUNDOS_POR_ANIO:     # menos de un año
        return f"{10 ** (exp_segundos - 4.94):.0f} días"
    if exp_segundos < 10.5:
        return f"{10 ** (exp_segundos - EXP_SEGUNDOS_POR_ANIO):,.0f} años".replace(",", ".")
    if exp_segundos < EXP_EDAD_UNIVERSO:
        return f"10^{exp_segundos - EXP_SEGUNDOS_POR_ANIO:.0f} años"
    return f"10^{exp_segundos - EXP_EDAD_UNIVERSO:,.0f} edades del universo".replace(",", ".")


cabecera = "{:36} {:>12} {:>14}   {}".format(
    "modelo", "perillas", "pruebas", "cuánto tardaría")
print(cabecera)
print("-" * 92)
for nombre, k in modelos:
    exp_pruebas = 2 * k                          # 100^k = 10^(2k)
    perillas = format(k, ",").replace(",", ".")
    pruebas = "10^" + format(exp_pruebas, ",").replace(",", ".")
    print("{:36} {:>12} {:>14}   {}".format(
        nombre, perillas, pruebas, en_castellano(exp_pruebas - EXP_POR_SEGUNDO)))

print()
print("Para comparar: se estima que hay unos 10^80 átomos en el universo observable.")
print("Nuestra recta necesitó 10^4 pruebas. La red de 8 rampas, 10^16: eso ya son")
print("cuatro meses de computadora. La fuerza bruta se murió en la tercera línea.")

---

## 4. La pendiente, con una resta

No podemos mirar el paisaje entero. Pero sí podemos hacer algo mucho más
barato: **movernos un poquito y ver si subimos o bajamos**.

Es la niebla. Estás en una colina, hay niebla cerrada y no ves nada más allá de
tus pies. No podés ver dónde está el valle. Pero sí podés adelantar un pie diez
centímetros y sentir si el suelo está más alto o más bajo. Con eso alcanza para
dar el paso siguiente. Y con el paso siguiente, para dar el otro.

Eso, escrito en Python, es **una línea**.

In [ ]:
def derivada_numerica(f, x, h=1e-4):
    """¿Cuánto cambia f si me muevo un poquito? Probamos para los dos lados.

    No hay nada más que esto. El nombre elegante de esta resta es 'derivada'.
    """
    return (f(x + h) - f(x - h)) / (2 * h)


# Probémosla con una función donde ya sabemos la respuesta.
# Si f(x) = x², la pendiente en x=3 es 6 (te lo decimos nosotros; no hace falta
# que lo sepas: lo que importa es que la resta lo encuentra sola).
print("f(x) = x²  ->  pendiente en x=3, calculada con una resta:",
      round(derivada_numerica(lambda t: t ** 2, 3.0), 6))
print("f(x) = x²  ->  pendiente en x=0, calculada con una resta:",
      round(derivada_numerica(lambda t: t ** 2, 0.0), 6))
print()
print("En x=0 la pendiente da 0: es el fondo. Ahí es donde queremos terminar.")

In [ ]:
def gradiente(a, b, h=1e-4):
    """Las dos pendientes: una por cada perilla.

    Al par de pendientes se le dice GRADIENTE, y es simplemente
    'para dónde sube la loss si muevo cada perilla por separado'.
    """
    pendiente_a = (loss(a + h, b) - loss(a - h, b)) / (2 * h)
    pendiente_b = (loss(a, b + h) - loss(a, b - h)) / (2 * h)
    return pendiente_a, pendiente_b


punto_a, punto_b = -0.30, 0.40      # arriba a la izquierda del mapa
ga, gb = gradiente(punto_a, punto_b)

def leer_pendiente(nombre, pendiente):
    sube = "SUBE" if pendiente > 0 else "BAJA"
    hacia = "bajarla" if pendiente > 0 else "subirla"
    return (f"  Si subo '{nombre}' un poquito, la loss {sube} "
            f"({pendiente:+.4f} por unidad)  ->  hay que {hacia}")


print(f"Estoy parado en a = {punto_a}, b = {punto_b}")
print(f"La loss ahí vale {loss(punto_a, punto_b):.4f}")
print()
print(leer_pendiente("a", ga))
print(leer_pendiente("b", gb))
print()
print("El par (pendiente_a, pendiente_b) es el GRADIENTE, y apunta a la subida")
print("más empinada. Nosotros vamos exactamente al revés. Eso es todo.")
print()
print("Ese par de números es LA flecha gruesa de la visualización")
print("viz/valle-de-loss.html, y acabás de calcularlo vos con dos restas.")

---

## 5. El descenso por gradiente, en diez líneas

Ya tenemos todo. La receta entera es:

1. Pararse en algún lado (donde sea).
2. Preguntar para dónde sube.
3. Ir para el otro lado, un pasito.
4. Repetir.

El «un pasito» tiene nombre: **learning rate** (tasa de aprendizaje). Es cuánto
te movés cada vez. Lo vamos a romper a propósito en tres minutos.

**Esta es la celda más importante del curso hasta acá.** Está comentada línea
por línea. En la clase 4, estas diez líneas se van a convertir en dos palabras
de PyTorch, y va a estar bien que así sea — pero recién después de haberlas
escrito a mano.

In [ ]:
# ---------------------------------------------------------------------------
# EL DESCENSO POR GRADIENTE, COMPLETO
# ---------------------------------------------------------------------------
a, b = -0.30, 0.40          # 1. me paro en algún lado (elegido a dedo, lejos)
lr = 0.05                   #    el tamaño del pasito: la TASA DE APRENDIZAJE
pasos = 200

historia_loss = [loss(a, b)]   #  para poder dibujar la curva después
rastro = [(a, b)]              #  por dónde fue pasando (a, b)

for i in range(pasos):
    pendiente_a, pendiente_b = gradiente(a, b)   # 2. ¿para dónde SUBE?

    a = a - lr * pendiente_a                     # 3. voy al revés: por eso el MENOS
    b = b - lr * pendiente_b                     #    y un pasito: por eso el lr

    historia_loss.append(loss(a, b))             # 4. anoto y repito
    rastro.append((a, b))

    if i % 20 == 0:
        print(f"paso {i:3d}   a = {a:+.4f}   b = {b:+.4f}   loss = {loss(a, b):.6f}")

print(f"paso {pasos:3d}   a = {a:+.4f}   b = {b:+.4f}   loss = {loss(a, b):.6f}")
print()
print(f"La fuerza bruta (14.400 pruebas) había encontrado a = {mejor_a:.4f}, b = {mejor_b:.4f}")
print(f"El descenso ({pasos} pasos) encontró          a = {a:.4f}, b = {b:.4f}")

In [ ]:
# Los tres gráficos que vas a ver el resto de tu vida
rastro_a = np.array([p[0] for p in rastro])
rastro_b = np.array([p[1] for p in rastro])

fig, ejes = g.figura(14, 4.2, filas=1, columnas=3,
                     titulo="Descenso por gradiente sobre datos reales")

# (1) la curva de loss: lo único que vas a tener cuando entrenes de verdad
ejes[0].plot(historia_loss, color=g.ERROR, lw=2.5)
ejes[0].axhline(paisaje.min(), color=g.OK, ls="--", lw=1.4)
ejes[0].set_title("la curva de loss")
ejes[0].set_xlabel("iteración"); ejes[0].set_ylabel("loss")
ejes[0].annotate("el fondo", (pasos * .55, paisaje.min()), xytext=(0, 8),
                 textcoords="offset points", color=g.OK, fontweight="bold")

# (2) el rastro sobre el mapa
ejes[1].contourf(rejilla_a, rejilla_b, paisaje, levels=niveles, cmap="YlOrRd")
ejes[1].plot(rastro_a, rastro_b, "-o", color=g.PARAMETRO, lw=1.4, ms=3)
ejes[1].plot(rastro_a[0], rastro_b[0], "o", color=g.PARAMETRO, ms=9,
             markeredgecolor="white", markeredgewidth=2)
ejes[1].plot(mejor_a, mejor_b, "o", color=g.OK, ms=9,
             markeredgecolor="white", markeredgewidth=2)
ejes[1].set_title("el rastro sobre el paisaje")
ejes[1].set_xlabel("qué tan empinada"); ejes[1].set_ylabel("dónde arranca")
ejes[1].set_aspect("equal")   # la misma escala en las dos perillas que en la
                              # celda 15: si no, el valle redondo se ve elíptico

# (3) la recta que encontró, sobre los datos
g.plot_ajuste(anios, seguridad, modelo=lambda t: a * (t - 2001.5) / 10 + b,
              mostrar_errores=True, ax=ejes[2],
              xlabel="año", ylabel="% de palabras",
              etiqueta_modelo="lo que encontró el descenso",
              titulo="la recta, en el mundo real")
plt.show()

print("Mirá el rastro del panel del medio: los puntitos arrancan separados y")
print("terminan pegados. Nadie frenó la pelota. Se frenó sola, porque la")
print("pendiente se aplana cerca del fondo y el paso se achica solo.")

---

## 6. Romperlo a propósito

Acá viene la parte útil. Vas a ver dos fracasos, y los dos tienen una **firma
visual** en la curva de loss. Aprender a reconocer esas dos formas te ahorra
tardes enteras en la clase 4 y en la 8.

Primero envolvemos el loop en una función, para poder llamarlo con distintos
valores sin copiar y pegar. **Es exactamente el mismo código de arriba.**

In [ ]:
def descenso(lr, pasos=200, a0=-0.30, b0=0.40):
    """El mismo loop de arriba, para poder probarlo con distintos valores."""
    a, b = a0, b0
    historia = [loss(a, b)]        # el punto de partida también cuenta
    for _ in range(pasos):
        pendiente_a, pendiente_b = gradiente(a, b)
        a = a - lr * pendiente_a
        b = b - lr * pendiente_b
        historia.append(loss(a, b))
    return a, b, np.array(historia)


# control: el que ya sabemos que anda
a_ok, b_ok, hist_ok = descenso(lr=0.05)
print(f"lr = 0.05   ->   a = {a_ok:.4f}, b = {b_ok:.4f}, loss = {hist_ok[-1]:.6f}")

### `TODO 3` — rompelo con un paso gigante

**Consigna.** Corré la celda de abajo tal cual está (`lr = 1.5`) y mirá los
primeros valores de `a`. Después escribí en un comentario, en **una** oración,
qué pasó.

**Pista.** Mirá los valores de `a` en las primeras cinco iteraciones y prestá
atención al **signo**. No están creciendo: están rebotando de un lado al otro,
cada vez más lejos.

**Aviso.** Si en algún momento ves `inf` o `nan`: felicitaciones, lo rompiste
bien. Eso es lo que tenía que pasar. `nan` quiere decir *not a number*, y es la
manera que tiene la computadora de decirte «este número se me fue de las
manos».

In [ ]:
# TODO 3: corré esto y escribí acá abajo, en una oración, qué pasó.
#
# Qué pasó: ...........................................................

a_mal, b_mal, hist_mal = descenso(lr=1.5, pasos=40)

# los primeros cinco pasos, para ver el rebote
a_v, b_v = -0.30, 0.40
print("Los primeros cinco pasos con lr = 1.5:")
for i in range(5):
    pa, pb = gradiente(a_v, b_v)
    a_v, b_v = a_v - 1.5 * pa, b_v - 1.5 * pb
    print(f"  paso {i + 1}:  a = {a_v:+12.4f}   loss = {loss(a_v, b_v):.4g}")

print()
digitos = len(f"{hist_mal[-1]:.0f}")
print(f"Después de 40 pasos la loss vale {hist_mal[-1]:.3g}: un número de {digitos} dígitos.")
print(f"Recordá que el fondo del valle estaba en {paisaje.min():.4f}.")
print()
print("Con MÁS pasos, acá el número se estaciona y deja de crecer. No es que se haya")
print("arreglado: es que nuestra derivada-por-resta se rompe con números tan grandes.")
print("Está explicado abajo. En un entrenamiento de verdad, con derivadas exactas, esto")
print("sigue creciendo hasta `inf` y después `nan` ('not a number'), que es la manera que")
print("tiene la computadora de decir 'este número se me fue de las manos'.")

revisar(hist_mal[-1] > hist_ok[-1] * 10,
        "Perfecto: lo rompiste. El paso era tan grande que en vez de bajar al "
        "valle, saltaba al otro lado, más arriba que antes. Y así cada vez.",
        "Con este learning rate no explotó. Subilo más y volvé a probar.")

> **Una honestidad, de paso.** Si dejás correr ese descenso roto muchos pasos
> más, vas a ver que el número **se congela** en vez de seguir creciendo. No es
> que se haya estabilizado: es que nuestra derivada-por-resta se rompe cuando
> los números son gigantescos, porque `loss(a + h)` y `loss(a - h)` dejan de
> distinguirse entre sí. Esa es una de las razones por las que las bibliotecas
> de verdad **no** calculan la derivada con restas: usan *backpropagation*, que
> da la derivada exacta y de una sola pasada. En la clase 4 son dos palabras:
> `loss.backward()`.

### `TODO 4` — rompelo con un paso microscópico

**Consigna.** Ahora al revés: `lr = 0.000001`. ¿Llegó? Estimá **cuántas
iteraciones harían falta** para llegar al fondo con ese paso.

**Pista.** Mirá cuánto bajó la loss en 200 pasos, y hacé una regla de tres con
lo que le falta bajar. Es una estimación grosera y está bien que lo sea.

In [ ]:
# TODO 4: corré y contestá abajo.
#
# ¿Llegó? ..............
# ¿Cuántas iteraciones harían falta, más o menos? ..............

a_lento, b_lento, hist_lento = descenso(lr=0.000001, pasos=200)

arranque = loss(-0.30, 0.40)
fondo = paisaje.min()
bajo_en_200 = arranque - hist_lento[-1]
le_falta = hist_lento[-1] - fondo

print(f"Arrancó en          {arranque:.6f}")
print(f"Después de 200 pasos {hist_lento[-1]:.6f}")
print(f"El fondo está en    {fondo:.6f}")
print()
print(f"Bajó {bajo_en_200:.8f} en 200 pasos. Le falta bajar {le_falta:.6f}.")
if bajo_en_200 > 0:
    faltan = le_falta / (bajo_en_200 / 200)
    print(f"A ese ritmo le faltarían del orden de {faltan:,.0f} iteraciones más."
          .replace(",", "."))
    print("(Es una regla de tres grosera: en realidad va frenando. Pero el orden")
    print(" de magnitud es el que importa, y el orden de magnitud es 'un montón'.)")

revisar(hist_lento[-1] > fondo * 2,
        "Ni cerca. No es que el método falle: es que el paso es tan chico que "
        "no llega en esta vida. Esta es la primera de las dos firmas que hay "
        "que reconocer: una curva de loss casi plana y ALTA.",
        "Bajó más de lo esperado. ¿Le cambiaste el lr?")

In [ ]:
# Las tres firmas, una al lado de la otra. Aprendete estas tres formas.
fig, ejes = g.figura(14, 3.8, filas=1, columnas=3,
                     titulo="Las tres firmas de la curva de loss")

# ojo con el tercer panel: su eje vertical llega CUATRO VECES más alto que los
# otros dos, y aun así la curva se le sale por arriba. Está dicho en el título.
paneles = [
    ("lr muy chico (0.000001)\ncasi plana y ALTA", hist_lento, g.VAL, 200, 0.25),
    ("lr bien (0.05)\ncae y se aplana ABAJO", hist_ok, g.OK, 200, 0.25),
    ("lr muy grande (1.5)\nse va para ARRIBA (ojo: otra escala)", hist_mal, g.ERROR, 20, 1.0),
]

for eje, (titulo, hist, color, xmax, techo) in zip(ejes, paneles):
    # lo que no entra en el cuadro se recorta, y se avisa con una flecha
    recortada = np.clip(np.nan_to_num(hist, nan=techo * 2, posinf=techo * 2), 0, techo * 2)
    eje.plot(recortada, color=color, lw=2.5)
    eje.axhline(fondo, color=g.OK, ls="--", lw=1.2)
    eje.set_ylim(0, techo)
    eje.set_xlim(0, xmax)
    eje.set_title(titulo, fontsize=10)
    eje.set_xlabel("iteración"); eje.set_ylabel("loss")

    se_escapa = np.flatnonzero(recortada > techo)
    if se_escapa.size:
        i = int(se_escapa[0])
        eje.annotate("se va para arriba", xy=(i, techo * .97),
                     xytext=(xmax * .25, techo * .62),
                     arrowprops=dict(arrowstyle="->", color=g.ERROR, lw=2.2),
                     color=g.ERROR, fontweight="bold", fontsize=10)
plt.show()

print("La del medio es la que querés ver. Las otras dos te dicen para qué lado")
print("mover el learning rate. Regla práctica: si sube, dividí el lr por 10.")

### `TODO 5` — arrancá desde muy lejos

**Consigna.** Corré el descenso arrancando desde `a = -5`, `b = 50`: un lugar
absurdo, muy fuera del mapa que dibujamos. ¿Llega al mismo lugar?

**Pista.** Necesitás más pasos que antes. Probá con 400.

**Lo que estás por descubrir.** En este paisaje, arranques donde arranques,
terminás en el mismo fondo, porque hay un solo pozo. **Eso no es una propiedad
general del descenso por gradiente: es una propiedad de este paisaje.** Abrí
`viz/valle-de-loss.html` y elegí «uno con dos pozos» para ver qué pasa cuando
hay dos. Esa diferencia es el tema de la última slide de hoy.

In [ ]:
# TODO 5: probá desde otros puntos de partida y anotá dónde termina cada uno.
a_lejos, b_lejos, hist_lejos = descenso(lr=0.05, pasos=400, a0=-5.0, b0=50.0)

print(f"Arrancando en (a=-5, b=50):  a = {a_lejos:.4f}, b = {b_lejos:.4f}")
print(f"Arrancando en (a=-0.3, b=0.4): a = {a_ok:.4f}, b = {b_ok:.4f}")
print(f"La fuerza bruta decía:         a = {mejor_a:.4f}, b = {mejor_b:.4f}")

llegan_al_mismo = abs(a_lejos - a_ok) < 0.01 and abs(b_lejos - b_ok) < 0.01
revisar(llegan_al_mismo,
        "Llegan al mismo lugar. Este paisaje tiene UN solo pozo, así que da lo "
        "mismo desde dónde arranques. Guardá esta frase: no siempre es así.",
        "Todavía no llegaron al mismo punto: probá con más pasos (600, 800). "
        "Desde tan lejos, el principio del camino es muy lento.")

---

## 7. Bonus: censo o muestra (mini-batches)

Hasta acá, cada paso usó **los 36 años**. Es un censo: la pendiente que
calculamos es la verdadera pendiente de la loss sobre todos los datos.

Con 36 casos eso es gratis. Con 36 millones de titulares, no: cada paso te
costaría minutos. La solución es de manual de metodología: **en vez de un
censo, una muestra**. Cada paso mira 5 años elegidos al azar y calcula la
pendiente con esos. La pendiente sale ruidosa —es una muestra— pero sale
muchísimo más barata, y con suficientes pasos llegás igual.

A esa muestra se le dice **mini-batch**, y al método, *descenso por gradiente
estocástico* (**SGD**, por *stochastic gradient descent*). «Estocástico»
quiere decir, exactamente, «con azar adentro».

In [ ]:
rng = np.random.default_rng(1983)      # semilla fija, siempre


def gradiente_de_muestra(a, b, indices, h=1e-4):
    """La misma cuenta de antes, pero mirando sólo algunos años."""
    xm, ym = x[indices], y[indices]

    def loss_m(aa, bb):
        return np.mean((ym - (aa * xm + bb)) ** 2)

    return ((loss_m(a + h, b) - loss_m(a - h, b)) / (2 * h),
            (loss_m(a, b + h) - loss_m(a, b - h)) / (2 * h))


a_mb, b_mb = -0.30, 0.40
rastro_mb, hist_mb = [], []
for _ in range(200):
    muestra = rng.choice(len(x), size=5, replace=False)     # 5 años al azar
    pa, pb = gradiente_de_muestra(a_mb, b_mb, muestra)
    a_mb, b_mb = a_mb - 0.05 * pa, b_mb - 0.05 * pb
    rastro_mb.append((a_mb, b_mb))
    hist_mb.append(loss(a_mb, b_mb))     # la loss REAL, sobre todos los datos

fig, ejes = g.figura(12, 4.4, filas=1, columnas=2,
                     titulo="Censo (todos los años) contra muestra (5 años por paso)")

ejes[0].contourf(rejilla_a, rejilla_b, paisaje, levels=niveles, cmap="YlOrRd")
ejes[0].plot(rastro_a, rastro_b, "-", color=g.MODELO, lw=2, label="todos los datos")
ejes[0].plot([p[0] for p in rastro_mb], [p[1] for p in rastro_mb], "-",
             color=g.VAL, lw=1.4, alpha=.9, label="5 al azar por paso")
ejes[0].plot(mejor_a, mejor_b, "o", color=g.OK, ms=9,
             markeredgecolor="white", markeredgewidth=2)
ejes[0].legend(loc="upper right", fontsize=9)
ejes[0].set_title("los dos rastros")
ejes[0].set_xlabel("qué tan empinada"); ejes[0].set_ylabel("dónde arranca")

ejes[1].plot(hist_ok, color=g.MODELO, lw=2, label="todos los datos")
ejes[1].plot(hist_mb, color=g.VAL, lw=1.4, label="5 al azar por paso")
ejes[1].axhline(fondo, color=g.OK, ls="--", lw=1.2)
ejes[1].set_title("las dos curvas de loss")
ejes[1].set_xlabel("iteración"); ejes[1].set_ylabel("loss")
ejes[1].legend(fontsize=9)
plt.show()

print(f"Con todos los datos : loss final {hist_ok[-1]:.6f}")
print(f"Con 5 al azar       : loss final {hist_mb[-1]:.6f}")
print()
print("El naranja tiembla. Ese temblor es el error de muestreo, y no es un")
print("defecto: es muchísimo más barato y, encima, a veces ayuda a escaparse")
print("de un pozo malo. De ahí sale la palabra 'estocástico'.")

---

### Y una última: por qué centramos el año

En la primera celda dividimos el año por 10 y le restamos 2001,5. Si no lo
hubiéramos hecho, la altura y la inclinación quedarían tan pegadas una a la
otra que el valle sería una **zanja diagonal** en vez de un pozo redondo. Y en
una zanja, el descenso zigzaguea de pared a pared y tarda muchísimo.

Corré esto y mirá la diferencia. **Es la razón por la que casi todo el mundo
estandariza sus variables antes de entrenar**, y ahora sabés cuál es: no es
higiene estadística, es la forma del valle.

In [ ]:
x_crudo = anios.astype(float)          # el año tal cual: 1984, 1985, ...


def loss_cruda(a, b):
    return np.mean((y - (a * x_crudo + b)) ** 2)


fig, ejes = g.figura(12, 4.4, filas=1, columnas=2,
                     titulo="La misma loss, con el año centrado y sin centrar")

# Los dos paneles usan LOS MISMOS niveles de color (del fondo del valle a veinte
# veces el fondo), así la comparación es honesta: lo que cambia es la forma.
mismos_niveles = np.geomspace(paisaje.min(), paisaje.min() * 20, 22)

# centrado: pozo redondo
ejes[0].contourf(rejilla_a, rejilla_b, paisaje, levels=mismos_niveles,
                 cmap="YlOrRd", extend="max")
ejes[0].plot(mejor_a, mejor_b, "o", color=g.OK, ms=8,
             markeredgecolor="white", markeredgewidth=2)
ejes[0].set_title("con el año centrado: un pozo redondo")
ejes[0].set_xlabel("qué tan empinada"); ejes[0].set_ylabel("dónde arranca")

# sin centrar: zanja diagonal. La ventana está puesta donde de verdad está el
# fondo: con el año crudo, la altura del fondo es -8,33 (el valor en el año 0).
ra = np.linspace(0.0004, 0.0080, 220)
rb = np.linspace(-17.0, 1.0, 220)
zanja = np.array([[loss_cruda(aa, bb) for aa in ra] for bb in rb])
ejes[1].contourf(ra, rb, zanja, levels=mismos_niveles, cmap="YlOrRd", extend="max")
fila_z, col_z = np.unravel_index(np.argmin(zanja), zanja.shape)
ejes[1].plot(ra[col_z], rb[fila_z], "o", color=g.OK, ms=8,
             markeredgecolor="white", markeredgewidth=2)
ejes[1].set_title("con el año crudo (1984, 1985, …): una zanja diagonal")
ejes[1].set_xlabel("qué tan empinada"); ejes[1].set_ylabel("dónde arranca")
plt.show()

print("En la zanja de la derecha, el gradiente casi nunca apunta al fondo:")
print("apunta a la pared de enfrente. Por eso el descenso zigzaguea.")
print("Ojo con la confusión clásica: el gradiente NO apunta al mínimo.")
print("Apunta a la subida más empinada ACÁ NOMÁS, que no es lo mismo.")

---

## 8. Qué te llevás

**Las tres cosas de hoy:**

1. **La loss es una decisión tuya, no un default del software.** MSE y MAE
   miran los mismos datos y le hacen caso a casos distintos. Elegir una es
   declarar qué error te duele más, y eso es una decisión metodológica —y a
   veces política— que ningún paquete estadístico puede tomar por vos.
2. **El gradiente es una resta.** «Me muevo un poquito para allá: ¿sube o
   baja?». No hace falta cálculo para entenderlo ni para calcularlo. Todo el
   aprendizaje automático es repetir esa pregunta y caminar al revés de la
   respuesta.
3. **La curva de loss se lee.** Casi plana y alta: el paso es muy chico. Cae y
   se aplana abajo: bien. Sube: el paso es muy grande, dividilo por diez. Esa
   es toda la artesanía, y la vas a usar desde la clase que viene hasta el
   final.

**El vocabulario nuevo** (también está en `recursos/glosario.md`):

| término | en castellano llano |
|---|---|
| *loss* / *loss function* | pérdida — el número que dice qué tan mal la estás dibujando |
| *MSE* | error cuadrático medio — el promedio de los errores al cuadrado |
| *MAE* | error absoluto medio — el promedio de los errores, sin signo |
| *gradient* | gradiente — para dónde sube la loss si movés cada perilla |
| *gradient descent* | descenso por gradiente — ir al revés del gradiente, a pasitos |
| *learning rate* | tasa de aprendizaje — de qué tamaño es cada pasito |
| *mini-batch* | lote — la muestra de datos que mira cada paso |
| *SGD* | descenso estocástico — el descenso, pero con muestras en vez de censo |
| *backpropagation* | la manera eficiente de calcular todos los gradientes de una pasada |

**Para el lunes, en tu propia investigación.** La próxima vez que leas «el
modelo fue optimizado», anotá al margen una pregunta: **¿optimizado para
minimizar qué?** Si la respuesta es «el default», ahí hay una decisión escondida
que nadie discutió. Es una pregunta de revisor que casi nadie hace y que casi
siempre encuentra algo.

**Lo que queda prometido.** Hoy calculamos los gradientes con restas, una perilla
por vez. Con dos perillas es gratis; con un millón y medio, no. Existe un truco
—**backpropagation**— que calcula *todos* los gradientes de una sola pasada.
No lo vamos a derivar. En la clase 4 lo vas a usar y son dos palabras:
`loss.backward()`.

---

### Para seguir

Las referencias de la clase están en `referencias.md`. Y si te quedaste con
ganas de mover cosas, las cuatro visualizaciones de hoy funcionan sin internet:

- `viz/comparador-de-losses.html` — arrastrá un caso y mirá girar la recta azul.
- `viz/valle-de-loss.html` — el mapa y la recta, sincronizados. Probá «uno con dos pozos».
- `viz/loss-1d.html` — la derivada, con una resta.
- `viz/tasa-de-aprendizaje.html` — las tres firmas, y un quiz para aparearlas.